# 05b — Rainfall Triggering: the *proper* Iverson error-function solution

**Short course:** *Geomorphological Hazards of Slopes* &nbsp;•&nbsp; University of Silesia in Sosnoviec &nbsp;•&nbsp; 2 ECTS

*Lecturer: Ola Fredin*

---

Notebook 05 used a first-order **exponential approximation** for the pore-pressure response to rainfall — convenient, analytically light, and within ~15 % of the truth, but not the actual Iverson (2000) solution. This companion notebook redoes the same calculations with the **proper error-function response** that comes from solving the 1-D linear diffusion equation exactly. Everything else — the FS time series, the interactive explorer, the multi-block Carpathian storm — is reproduced so the two notebooks can be compared side by side.

## 1. The proper Iverson response function

Iverson (2000) models the pressure-head response of a wet soil to vertical infiltration as 1-D linear diffusion,

$$
\frac{\partial \psi}{\partial t} = D\,\frac{\partial^2 \psi}{\partial z^2},
\qquad D = D_0\cos^2\beta .
$$

A **constant surface flux** $I_Z$ switched on at $t=0$ over a semi-infinite column is the classic *constant-flux-into-a-half-space* problem. Its exact solution for the pressure-head rise at depth $z$ is

$$
\Delta\psi(z,t) = \frac{I_Z\cos^2\beta}{K_z}\,z\,R(t_\star),
\qquad
R(t_\star) = \sqrt{\frac{t_\star}{\pi}}\,e^{-1/t_\star} - \operatorname{erfc}\!\left(\frac{1}{\sqrt{t_\star}}\right),
\qquad
t_\star = \frac{4 D\,t}{z^{2}} .
$$

This is the genuine Iverson response function, built from the **complementary error function** $\operatorname{erfc}$ — no exponential short-cut. Two things to note versus notebook 05:

- $R(t_\star)\to 0$ as $t_\star\to 0$ (no instantaneous response), exactly as before.
- $R(t_\star)$ keeps **growing** like $\sqrt{t_\star}$ for a sustained input — the half-space never reaches a finite steady state. The exponential model artificially plateaus at $\Delta\psi_\infty$; the proper solution does not.

For a storm of finite duration $T$ we superpose a *negative* infiltration starting at $t=T$ (turning the rain off):

$$
\Delta\psi(z,t) = \frac{I_Z\cos^2\beta}{K_z}\,z \times
\begin{cases}
R(t_\star), & 0\le t\le T,\\[4pt]
R(t_\star) - R(t_\star - T_\star), & t> T,
\end{cases}
\qquad T_\star = \frac{4 D\,T}{z^{2}} .
$$

The difference of the two square-root terms decays to zero, so after the storm the pressure correctly drains away.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erfc

from style import apply_style, COLORS, save_figure
apply_style()

from ipywidgets import interact, FloatSlider

GAMMA_W = 9.81  # unit weight of water [kN/m^3]


def iverson_R(t_star):
    """Iverson (2000) response function R(t*) = sqrt(t*/pi) e^{-1/t*} - erfc(1/sqrt(t*)).

    Built from the complementary error function (the proper solution),
    not the exponential approximation of notebook 05. R(0) = 0.
    """
    t_star = np.asarray(t_star, dtype=float)
    R = np.zeros_like(t_star)
    pos = t_star > 0.0
    ts = t_star[pos]
    R[pos] = np.sqrt(ts / np.pi) * np.exp(-1.0 / ts) - erfc(1.0 / np.sqrt(ts))
    return R


def delta_u_iverson(t, T_storm, z, beta_deg, I_mm_h, K_mm_h, D_m2_per_s):
    """Pore-pressure response Delta_u(t) at depth z for a storm of duration T_storm,
    using the proper Iverson error-function response.

    Parameters as in notebook 05 (times in seconds). Returns Delta_u in kPa.
    """
    beta = np.radians(beta_deg)
    I_m_s = (I_mm_h / 1000.0) / 3600.0
    K_m_s = (K_mm_h / 1000.0) / 3600.0
    I_eff = min(I_m_s, K_m_s)                      # cap at conductivity (no runoff)
    coef = I_eff * np.cos(beta) ** 2 / K_m_s * z   # = Delta_psi_inf prefactor [m]

    t = np.asarray(t, dtype=float)
    t_star = 4.0 * D_m2_per_s * t / z ** 2
    T_star = 4.0 * D_m2_per_s * T_storm / z ** 2

    psi = np.where(
        t <= T_storm,
        coef * iverson_R(t_star),
        coef * (iverson_R(t_star) - iverson_R(t_star - T_star)),
    )
    return GAMMA_W * psi   # kPa


## 2. Response to a 48-hour storm

Same slip surface and storm as notebook 05 ($z = 2$ m, $\beta = 28^\circ$, $I = 4$ mm/h for 48 h), now with the error-function response.

In [ ]:
T_storm_h = 48.0
t_h = np.linspace(0, 168, 500)          # one week in hours
t_s = t_h * 3600.0
T_storm_s = T_storm_h * 3600.0

du = delta_u_iverson(
    t=t_s, T_storm=T_storm_s,
    z=2.0, beta_deg=28.0,
    I_mm_h=4.0, K_mm_h=8.0, D_m2_per_s=2e-5,
)

fig, ax = plt.subplots()
ax.fill_between([0, T_storm_h], 0, max(du) * 1.1, color=COLORS["water"], alpha=0.15,
                label=f"storm ({T_storm_h:.0f} h)")
ax.plot(t_h, du, color=COLORS["water"], lw=2.2,
        label=r"$\Delta u(z=2\,\mathrm{m}, t)$  (Iverson erf)")
ax.set_xlabel("time since storm onset  [h]")
ax.set_ylabel(r"pore-pressure rise  $\Delta u$  [kPa]")
ax.set_title("Iverson error-function response of a 2-m slip surface to 48 h of moderate rain")
ax.legend(loc="upper right")
save_figure(fig, "rainfall_pore_pressure_response_iverson")
plt.show()

print(f"Peak Delta u   ~ {du.max():.2f} kPa")
print(f"Peak occurs at ~ {t_h[np.argmax(du)]:.1f} h after storm onset (storm ends at {T_storm_h:.0f} h)")


## 3. From $\Delta u(t)$ to FS(t)

Identical machinery to notebook 05: convert pore pressure to a saturation fraction $m(t)$ and feed it into the infinite-slope factor of safety. Only the infiltration response has changed.

In [ ]:
def fs_infinite_slope(beta_deg, z, c_prime, phi_deg, gamma, m):
    """Same FS function as notebooks 04 and 05."""
    beta = np.radians(beta_deg); phi = np.radians(phi_deg)
    return (c_prime / (gamma * z * np.sin(beta) * np.cos(beta))
            + (1.0 - m * GAMMA_W / gamma) * np.tan(phi) / np.tan(beta))


# Carpathian flysch slope from notebooks 03-05
beta_deg, z, c_prime, phi_deg, gamma = 28.0, 2.0, 5.0, 28.0, 19.0
m0 = 0.30
u0 = m0 * GAMMA_W * z * np.cos(np.radians(beta_deg))**2

m_t = (u0 + du) / (GAMMA_W * z * np.cos(np.radians(beta_deg))**2)
FS_t = np.array([fs_infinite_slope(beta_deg, z, c_prime, phi_deg, gamma, m) for m in m_t])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.0, 6.4), sharex=True)
ax1.fill_between([0, T_storm_h], 0, m_t.max()*1.1, color=COLORS["water"], alpha=0.15)
ax1.plot(t_h, m_t, color=COLORS["water"], lw=2.2)
ax1.axhline(m0, color=COLORS["neutral"], lw=1.0, ls=":")
ax1.text(170, m0 + 0.01, fr"$m_0$ = {m0:.2f}", color=COLORS["neutral"], fontsize=11, ha="right")
ax1.set_ylabel(r"saturation fraction  $m$")
ax1.set_title(fr"Slope response to a 48 h storm (Iverson erf)  ($\beta$ = {beta_deg:.0f}$^\circ$, "
              fr"$z$ = {z:.1f} m,  $c'$ = {c_prime:.0f} kPa)")

ax2.fill_between([0, T_storm_h], 0, FS_t.max()*1.1, color=COLORS["water"], alpha=0.15)
ax2.plot(t_h, FS_t, color=COLORS["fail"], lw=2.2)
ax2.axhline(1.0, color="black", lw=1.0, ls="--")
ax2.text(170, 1.03, "FS = 1", color="black", fontsize=11, ha="right")

below = np.where(FS_t < 1.0)[0]
if below.size:
    t_fail = t_h[below[0]]
    ax2.plot([t_fail], [1.0], "o", ms=10, color=COLORS["fail"])
    ax2.annotate(fr"failure at $t$ = {t_fail:.1f} h",
                 xy=(t_fail, 1.0), xytext=(15, -15),
                 textcoords="offset points", color=COLORS["fail"], fontsize=11)

ax2.set_xlabel("time since storm onset  [h]")
ax2.set_ylabel("factor of safety  FS")
ax2.set_ylim(0.5, max(FS_t.max(), 1.4))
save_figure(fig, "fs_time_series_iverson")
plt.show()


## 4. Interactive exploration

The same controls as notebook 05. Compare the curves with the exponential version: the proper response rises a little faster at first (timescale $z^2/4D$ rather than $z^2/D$) and, for long storms, keeps climbing instead of plateauing at $\Delta\psi_\infty$.

In [ ]:
def explore(I_mm_h=4.0, T_storm_h=48.0, K_mm_h=8.0, D_m2_per_s_log10=-4.7,
            z=2.0, beta_deg=28.0, c_prime=5.0, phi_deg=28.0, gamma=19.0, m0=0.30):
    D_m2_per_s = 10**D_m2_per_s_log10
    t_h = np.linspace(0, max(168, T_storm_h * 4), 400)
    t_s = t_h * 3600.0
    du = delta_u_iverson(t_s, T_storm_h * 3600.0, z, beta_deg, I_mm_h, K_mm_h, D_m2_per_s)
    u0 = m0 * GAMMA_W * z * np.cos(np.radians(beta_deg))**2
    m_t = (u0 + du) / (GAMMA_W * z * np.cos(np.radians(beta_deg))**2)
    FS_t = np.array([fs_infinite_slope(beta_deg, z, c_prime, phi_deg, gamma, m) for m in m_t])

    fig, (a1, a2) = plt.subplots(2, 1, figsize=(8.5, 5.6), sharex=True)
    a1.fill_between([0, T_storm_h], 0, du.max() * 1.2 + 1, color=COLORS["water"], alpha=0.15)
    a1.plot(t_h, du, color=COLORS["water"], lw=2.0)
    a1.set_ylabel(r"$\Delta u$  [kPa]")
    a1.set_title(fr"I = {I_mm_h:.1f} mm/h x {T_storm_h:.0f} h,  K = {K_mm_h:.1f} mm/h,  D = $10^{{{D_m2_per_s_log10:.1f}}}$ m$^2$/s")

    a2.fill_between([0, T_storm_h], 0, max(FS_t.max() * 1.05, 1.3), color=COLORS["water"], alpha=0.15)
    a2.plot(t_h, FS_t, color=COLORS["fail"], lw=2.0)
    a2.axhline(1.0, color="black", lw=1.0, ls="--")
    a2.set_ylabel("FS")
    a2.set_xlabel("time  [h]")
    a2.set_ylim(0.4, max(FS_t.max() * 1.05, 1.4))
    plt.show()


interact(
    explore,
    I_mm_h          = FloatSlider(min=0.5, max=30.0, step=0.5, value=4.0,  description="I [mm/h]"),
    T_storm_h       = FloatSlider(min=1.0, max=120.0, step=1.0, value=48.0, description="T [h]"),
    K_mm_h          = FloatSlider(min=1.0, max=40.0, step=1.0, value=8.0,  description="K [mm/h]"),
    D_m2_per_s_log10= FloatSlider(min=-6.0, max=-3.5, step=0.1, value=-4.7, description="log10 D"),
);


## 5. Worked example: a multi-day Carpathian rainstorm

The same three-block storm as notebook 05, superposed linearly (each pulse rises and decays via its own Iverson response).

In [ ]:
events = [
    dict(start=0.0,   duration=18.0, I=6.0),
    dict(start=30.0,  duration=24.0, I=3.0),
    dict(start=72.0,  duration=12.0, I=10.0),
]

t_h = np.linspace(0, 168, 1200)
t_s = t_h * 3600.0
du_total = np.zeros_like(t_h)
intensity = np.zeros_like(t_h)

for ev in events:
    t_shift = (t_h - ev["start"]).clip(min=0) * 3600.0
    du_total += delta_u_iverson(
        t=t_shift, T_storm=ev["duration"] * 3600.0,
        z=2.0, beta_deg=28.0, I_mm_h=ev["I"], K_mm_h=8.0, D_m2_per_s=2e-5,
    )
    mask = (t_h >= ev["start"]) & (t_h <= ev["start"] + ev["duration"])
    intensity[mask] += ev["I"]

u0 = m0 * GAMMA_W * z * np.cos(np.radians(beta_deg))**2
m_t = (u0 + du_total) / (GAMMA_W * z * np.cos(np.radians(beta_deg))**2)
FS_t = np.array([fs_infinite_slope(beta_deg, z, c_prime, phi_deg, gamma, m) for m in m_t])

fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(8.5, 7.0), sharex=True,
                                  gridspec_kw=dict(height_ratios=[0.7, 1.0, 1.0]))
a1.bar(t_h, intensity, width=t_h[1] - t_h[0], color=COLORS["water"], alpha=0.85)
a1.set_ylabel("I [mm/h]")
a1.set_title("Three-block Carpathian rainstorm (Iverson erf response)")

a2.plot(t_h, du_total, color=COLORS["water"], lw=2.2)
a2.set_ylabel(r"$\Delta u$  [kPa]")

a3.plot(t_h, FS_t, color=COLORS["fail"], lw=2.2)
a3.axhline(1.0, color="black", lw=1.0, ls="--")
below = np.where(FS_t < 1.0)[0]
if below.size:
    t_fail = t_h[below[0]]
    a3.plot([t_fail], [1.0], "o", ms=10, color=COLORS["fail"])
    a3.annotate(fr"failure at $t$ = {t_fail:.0f} h",
                xy=(t_fail, 1.0), xytext=(10, -20),
                textcoords="offset points", color=COLORS["fail"], fontsize=11)
a3.set_xlabel("time  [h]")
a3.set_ylabel("FS")
a3.set_ylim(0.5, max(FS_t.max(), 1.4))
save_figure(fig, "carpathian_storm_event_iverson")
plt.show()

print(f"Cumulative rainfall: {sum(e['I']*e['duration'] for e in events):.0f} mm")
print(f"Minimum FS:          {FS_t.min():.2f}")
print(f"First failure at:    {t_h[below[0]]:.0f} h" if below.size else "Slope did not fail.")


## 6. Exponential vs. error-function — side by side

Direct comparison of the two response functions for the single 48 h storm, to see exactly where the notebook-05 approximation deviates from the proper Iverson solution.

In [ ]:
def delta_u_exponential(t, T_storm, z, beta_deg, I_mm_h, K_mm_h, D_m2_per_s):
    """Notebook-05 exponential approximation, reproduced for comparison."""
    beta = np.radians(beta_deg)
    I_m_s = (I_mm_h / 1000.0) / 3600.0
    K_m_s = (K_mm_h / 1000.0) / 3600.0
    I_eff = min(I_m_s, K_m_s)
    delta_psi_inf = I_eff * np.cos(beta)**2 / K_m_s * z
    tau = z**2 / D_m2_per_s
    t = np.asarray(t, dtype=float)
    psi = np.where(
        t <= T_storm,
        delta_psi_inf * (1.0 - np.exp(-t / tau)),
        delta_psi_inf * (1.0 - np.exp(-T_storm / tau)) * np.exp(-(t - T_storm) / tau),
    )
    return GAMMA_W * psi


t_h = np.linspace(0, 168, 600)
t_s = t_h * 3600.0
kw = dict(T_storm=48*3600.0, z=2.0, beta_deg=28.0, I_mm_h=4.0, K_mm_h=8.0, D_m2_per_s=2e-5)
du_erf = delta_u_iverson(t_s, **kw)
du_exp = delta_u_exponential(t_s, **kw)

fig, ax = plt.subplots()
ax.fill_between([0, 48], 0, max(du_erf.max(), du_exp.max())*1.1,
                color=COLORS["water"], alpha=0.12, label="storm (48 h)")
ax.plot(t_h, du_erf, color=COLORS["fail"], lw=2.2, label="proper Iverson (erf)")
ax.plot(t_h, du_exp, color=COLORS["accent"], lw=2.2, ls="--", label="exponential (nb 05)")
ax.set_xlabel("time since storm onset  [h]")
ax.set_ylabel(r"pore-pressure rise  $\Delta u$  [kPa]")
ax.set_title("Proper error-function vs. exponential approximation")
ax.legend(loc="upper right")
save_figure(fig, "iverson_erf_vs_exponential")
plt.show()

print(f"Peak (erf): {du_erf.max():.2f} kPa at {t_h[np.argmax(du_erf)]:.1f} h")
print(f"Peak (exp): {du_exp.max():.2f} kPa at {t_h[np.argmax(du_exp)]:.1f} h")


## References

- Iverson, R. M. (2000). *Landslide triggering by rain infiltration.* Water Resources Research, 36(7), 1897-1910.
- Baum, R. L., Savage, W. Z., & Godt, J. W. (2008). *TRIGRS - A Fortran program for transient rainfall infiltration and grid-based regional slope-stability analysis, version 2.0.* USGS Open-File Report 2008-1159.
- Carslaw, H. S. & Jaeger, J. C. (1959). *Conduction of Heat in Solids*, 2nd ed. (constant-flux half-space solution.)